# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


In [5]:
print(df.columns.tolist())
print("Veri boyutu:", df.shape)

['review_id', 'length_review', 'review_score', 'order_id', 'product_category_name', 'full_review']
Veri boyutu: (36148, 6)


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [6]:

import re
import nltk
import unidecode
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("rslp")

stop_words = set(stopwords.words("portuguese"))
stemmer = RSLPStemmer()

def cleaning(sentence):
    if pd.isna(sentence):
        return ""

    sentence = unidecode.unidecode(str(sentence).lower())
    sentence = re.sub(r"\d+", " ", sentence)
    sentence = re.sub(r"[^a-z\s]", " ", sentence)

    tokens = word_tokenize(sentence)

    cleaned_tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(cleaned_tokens)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping stemmers\rslp.zip.


In [7]:

df["full_review_cleaned"] = df["full_review"].apply(cleaning)

df[["full_review", "full_review_cleaned"]].head()


,full_review,full_review_cleaned
0,Recebi bem antes do prazo estipulado.,receb bem ant praz estipul
1,Parabéns lojas lannister adorei comprar pela ...,parab loj lannist ador compr internet segur pr...
2,recomendo aparelho eficiente. no site a marca ...,recom aparelh efici sit marc aparelh impress d...
3,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouc trav val boa
4,"Super recomendo Vendedor confiável, produto ok...",sup recom vend confia produt entreg ant praz


## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [8]:

# 1, 2 ve 3 yıldızlı yorumları belirle
bad_reviews_mask = df["review_score"].between(1, 3)

# Kötü yorumların oranı
bad_reviews_ratio = bad_reviews_mask.mean()

print("Toplam yorum sayısı:", len(df))
print("Kötü yorum sayısı:", bad_reviews_mask.sum())
print("Kötü yorum oranı: %", round(bad_reviews_ratio * 100, 2))


Toplam yorum sayısı: 36148
Kötü yorum sayısı: 9658
Kötü yorum oranı: % 26.72


🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [9]:

# Sadece 1, 2 veya 3 yıldızlı yorumları al
bad_reviews = df.loc[bad_reviews_mask].copy()

print("Kötü yorumların boyutu:", bad_reviews.shape)

bad_reviews[
    ["review_score", "product_category_name", "full_review_cleaned"]
].head()


Kötü yorumların boyutu: (9658, 7)


,review_score,product_category_name,full_review_cleaned
14,1,eletronicos,receb soment control mide split estil falt con...
23,3,ferramentas_jardim,compr dua unidad receb agor fac
24,3,beleza_saude,produt bom por vei mim nao condiz fot anunci
25,1,moveis_decoracao,produt inferi mal acab
28,3,esporte_lazer,entreg praz


### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [10]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Tek kelimeleri ve bigramları kullan
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_df=0.75,
    max_features=5000
)

# Kötü yorumları sayısal verilere dönüştür
X = vectorizer.fit_transform(
    bad_reviews["full_review_cleaned"].fillna("")
)

print("Vektörleştirme tamamlandı!")
print("Veri boyutu:", X.shape)
print("Özellik sayısı:", len(vectorizer.get_feature_names_out()))


Vektörleştirme tamamlandı!
Veri boyutu: (9658, 5000)
Özellik sayısı: 5000


### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [11]:

from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(
    n_components=3,
    random_state=42
)

lda_model.fit(X)

print("LDA modeli başarıyla eğitildi!")
print("Konu sayısı:", lda_model.n_components)
print("Konu-kelime matrisi:", lda_model.components_.shape)


LDA modeli başarıyla eğitildi!
Konu sayısı: 3
Konu-kelime matrisi: (3, 5000)


#### Belge Karışımı (Konular için)

In [12]:

# Her kötü yorumun 3 konuya ait olasılıklarını hesapla
document_topic_mixture = lda_model.transform(X)

print("Belge-konu matrisi:", document_topic_mixture.shape)

# İlk 5 yorumun konu dağılımı
topic_df = pd.DataFrame(
    document_topic_mixture,
    columns=["Topic 0", "Topic 1", "Topic 2"]
)

topic_df.head()


Belge-konu matrisi: (9658, 3)


,Topic 0,Topic 1,Topic 2
0,0.697631,0.190431,0.111938
1,0.083952,0.188585,0.727463
2,0.526100,0.398930,0.074970
3,0.106892,0.296581,0.596526
4,0.139431,0.131579,0.728990


👉 Her inceleme için en önemli konuyu rapor edelim

In [13]:

import numpy as np

# Her kötü yorum için en baskın konuyu belirle
bad_reviews["most_important_topic"] = np.argmax(
    document_topic_mixture,
    axis=1
)

# Sonraki notebook bölümleri için tüm yorumlara da konu ata
all_reviews_vectorized = vectorizer.transform(
    df["full_review_cleaned"].fillna("")
)

df["most_important_topic"] = lda_model.transform(
    all_reviews_vectorized
).argmax(axis=1)

print("Kötü yorumlarda konu dağılımı:")
print(bad_reviews["most_important_topic"].value_counts())

bad_reviews[
    ["review_score", "full_review_cleaned", "most_important_topic"]
].head()


Kötü yorumlarda konu dağılımı:
most_important_topic
1    3691
0    3046
2    2921
Name: count, dtype: int64


,review_score,full_review_cleaned,most_important_topic
14,1,receb soment control mide split estil falt con...,0
23,3,compr dua unidad receb agor fac,2
24,3,produt bom por vei mim nao condiz fot anunci,0
25,1,produt inferi mal acab,2
28,3,entreg praz,2


#### Konu Karışımı (Kelimeler için)

In [14]:

# LDA modelindeki konu-kelime ağırlıkları
topic_word_mixture_matrix = lda_model.components_

print("Konu-kelime matrisinin boyutu:")
print(topic_word_mixture_matrix.shape)

# Her konunun en önemli 10 kelimesini göster
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(topic_word_mixture_matrix):
    top_indices = topic.argsort()[-10:][::-1]
    top_words = feature_names[top_indices]

    print(f"\nTopic {topic_idx}:")
    print(", ".join(top_words))


Konu-kelime matrisinin boyutu:
(3, 5000)

Topic 0:
bom, nao, produt, entreg, defeit, vei, cor, aind, otim, cheg

Topic 1:
receb, nao, produt, compr, nao receb, receb produt, entreg, outr, doi, ate

Topic 2:
produt, vei, cheg, nao, praz, entreg, recom, compr, boa, unidad


#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [15]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [16]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [17]:

print_topics(vectorizer, lda_model, 10)


--------------------
Topic 0:
[('bom', np.float64(233.25)), ('nao', np.float64(152.74)), ('produt', np.float64(134.3)), ('entreg', np.float64(102.19)), ('defeit', np.float64(79.59)), ('vei', np.float64(71.3)), ('cor', np.float64(67.92)), ('aind', np.float64(63.74)), ('otim', np.float64(62.3)), ('cheg', np.float64(62.12))]
--------------------
Topic 1:
[('receb', np.float64(295.42)), ('nao', np.float64(240.42)), ('produt', np.float64(225.97)), ('compr', np.float64(171.02)), ('nao receb', np.float64(170.02)), ('receb produt', np.float64(140.97)), ('entreg', np.float64(137.36)), ('outr', np.float64(90.25)), ('doi', np.float64(83.76)), ('ate', np.float64(82.6))]
--------------------
Topic 2:
[('produt', np.float64(151.3)), ('vei', np.float64(128.49)), ('cheg', np.float64(106.81)), ('nao', np.float64(98.3)), ('praz', np.float64(95.16)), ('entreg', np.float64(86.94)), ('recom', np.float64(84.1)), ('compr', np.float64(80.59)), ('boa', np.float64(75.12)), ('unidad', np.float64(71.6))]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [18]:

# Her konunun en önemli 10 kelimesini sakla
topic_word_mixture = {
    i: topic_word(
        vectorizer,
        lda_model,
        topic,
        10,
        with_weights=False
    )
    for i, topic in enumerate(lda_model.components_)
}

for topic, words in topic_word_mixture.items():
    print(f"Topic {topic}: {', '.join(words)}")


Topic 0: bom, nao, produt, entreg, defeit, vei, cor, aind, otim, cheg
Topic 1: receb, nao, produt, compr, nao receb, receb produt, entreg, outr, doi, ate
Topic 2: produt, vei, cheg, nao, praz, entreg, recom, compr, boa, unidad


In [19]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [20]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e64fb393e7b32834bb789ff8bb30750e,5,ferramentas_jardim,receb bem ant praz estipul,2,"[produt, vei, cheg, nao, praz, entreg, recom, ..."
1,f7c4243c7fe1938f181bec41a392bdeb,5,esporte_lazer,parab loj lannist ador compr internet segur pr...,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."
2,8670d52e15e00043ae7de4c01cc2fe06,4,eletroportateis,recom aparelh efici sit marc aparelh impress d...,1,"[receb, nao, produt, compr, nao receb, receb p..."
3,4b49719c8a200003f700d3d986ea1a19,4,beleza_saude,pouc trav val boa,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."
4,3948b09f7c818e2d86c9a546758b2335,5,informatica_acessorios,sup recom vend confia produt entreg ant praz,2,"[produt, vei, cheg, nao, praz, entreg, recom, ..."


## (3.4) Pipeline Tf-Idf ve LDA

In [21]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [22]:

from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        max_df=0.75,
        max_features=5000
    ),
    LatentDirichletAllocation(
        n_components=3,
        random_state=42
    )
)

pipeline.fit(df["full_review_cleaned"].fillna(""))

pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('latentdirichletallocation', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.75
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",5000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only 

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [23]:
pipeline._final_estimator

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"random_state random_state: int, RandomState instance or None, default=NonePass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0


In [24]:
pipeline._final_estimator.components_

array([[ 3.18006043,  0.33590857, 19.73504737, ...,  4.93607477,
         9.33433661,  8.64523387],
       [ 5.9972668 ,  3.32509031,  0.33961859, ...,  0.33875868,
         0.33976466,  0.33782847],
       [ 0.34859087,  0.34304024,  0.34552689, ...,  0.33761253,
         0.33512693,  0.33829646]], shape=(3, 5000))

Pipeline ile **Belge Karışımı**:

In [25]:

pipeline_document_topics = pipeline.transform(
    df["full_review_cleaned"].fillna("")
)

print("Belge-konu matrisi:", pipeline_document_topics.shape)

pd.DataFrame(
    pipeline_document_topics,
    columns=["Topic 0", "Topic 1", "Topic 2"]
).head()


Belge-konu matrisi: (36148, 3)


,Topic 0,Topic 1,Topic 2
0,0.095555,0.088763,0.815682
1,0.149518,0.674817,0.175665
2,0.797324,0.099395,0.103280
3,0.638121,0.185326,0.176553
4,0.076477,0.324222,0.599300


Pipeline ile **Konu Karışımı**:

In [26]:

pipeline_topic_words = pipeline._final_estimator.components_

print("Konu-kelime matrisi:", pipeline_topic_words.shape)

pipeline_vectorizer = pipeline.named_steps["tfidfvectorizer"]

print_topics(
    pipeline_vectorizer,
    pipeline._final_estimator,
    10
)


Konu-kelime matrisi: (3, 5000)
--------------------
Topic 0:
[('nao', np.float64(904.66)), ('receb', np.float64(622.44)), ('produt', np.float64(540.94)), ('compr', np.float64(508.63)), ('vei', np.float64(445.48)), ('ped', np.float64(276.49)), ('receb produt', np.float64(272.63)), ('aind', np.float64(247.74)), ('nao receb', np.float64(217.83)), ('entreg', np.float64(212.04))]
--------------------
Topic 1:
[('rap', np.float64(704.66)), ('tud', np.float64(644.45)), ('entreg', np.float64(505.3)), ('entreg rap', np.float64(399.28)), ('cert', np.float64(394.01)), ('atend', np.float64(306.32)), ('lind', np.float64(296.34)), ('produt', np.float64(256.56)), ('tud cert', np.float64(247.08)), ('qual', np.float64(243.42))]
--------------------
Topic 2:
[('bom', np.float64(1957.42)), ('otim', np.float64(1473.27)), ('recom', np.float64(1285.26)), ('praz', np.float64(1168.96)), ('produt', np.float64(1059.54)), ('ant', np.float64(958.25)), ('ant praz', np.float64(824.17)), ('excel', np.float64(800.05)

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [27]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                   \
                                                      count      mean median   
product_category_name                                                          
alimentos_bebidas                                        75  4.560000    5.0   
livros_interesse_geral                                  152  4.539474    5.0   
malas_acessorios                                        375  4.317333    5.0   
livros_tecnicos                                          83  4.301205    5.0   
pcs                                                      67  4.283582    5.0   
alimentos                                               151  4.231788    5.0   
papelaria                                               758  4.201847    5.0   
instrumentos_musicais                                   228  4.192982    5.0   
fashion_calcados                                        101  4.188119    5.0   
fashion_bolsas_e_acessorios                             726  4.168044    5.0   
perfumaria                                             1094  4.167276    5.0   
construcao_ferramentas_jardim                            85  4.164706    5.0   
cool_stuff                                             1360  4.154412    5.0   
ferramentas_jardim                                     1391  4.105679    5.0   
beleza_saude                                           3046  4.104071    5.0   
brinquedos                                             1271  4.101495    5.0   
esporte_lazer                                          2637  4.097838    5.0   
bebidas                                                 111  4.090090    5.0   
pet_shop                                                603  4.086235    5.0   
construcao_ferramentas_iluminacao                        86  4.081395    5.0   
eletroportateis                                         242  4.061983    5.0   
automotivo                                             1533  4.049576    5.0   
eletrodomesticos                                        354  4.033898    5.0   
eletrodomesticos_2                                       89  4.011236    5.0   
relogios_presentes                                     2211  4.009950    5.0   
construcao_ferramentas_construcao                       277  4.007220    5.0   
consoles_games                                          339  4.005900    5.0   
utilidades_domesticas                                  2192  4.005018    5.0   
eletronicos                                             942  3.979830    5.0   
industria_comercio_e_negocios                            82  3.963415    5.0   
bebes                                                   882  3.955782    5.0   
sinalizacao_e_seguranca                                  61  3.934426    5.0   
telefonia                                              1654  3.874244    5.0   
cama_mesa_banho                                        3891  3.853765    5.0   
artes                                                    80  3.850000    5.0   
casa_conforto                                           175  3.817143    5.0   
moveis_decoracao                                       2311  3.815232    5.0   
moveis_sala                                             138  3.789855    5.0   
informatica_acessorios                                 2473  3.774767    5.0   
audio                                                   134  3.768657    5.0   
market_place                                            102  3.745098    4.0   
moveis_cozinha_area_de_servico_jantar_e_jardim           94  3.734043    4.0   
construcao_ferramentas_seguranca                         61  3.721311    5.0   
agro_industria_e_comercio                                57  3.701754    4.0   
telefonia_fixa                                           96  3.697917    4.0   
casa_construcao                                         177  3.672316    4.0   
climatizacao                                             93  3.655914    4.0   
fashion_roupa_masculina                        

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [28]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                               count      mean median       std
product_category_name                                          
moveis_escritorio                546  3.298535    4.0  1.562366
casa_construcao                  177  3.672316    4.0  1.498094
telefonia_fixa                    96  3.697917    4.0  1.576938
climatizacao                      93  3.655914    4.0  1.625173
fashion_roupa_masculina           56  3.482143    4.0  1.737198

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [29]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
45,c422274e50e900b46fee429016c5458d,3,moveis_escritorio,gost,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."
126,5d6f9cddc8335878d8bf20c3bd4602e8,5,moveis_escritorio,meg recom receb produt corret dentr praz verif...,2,"[produt, vei, cheg, nao, praz, entreg, recom, ..."
212,1f836d160f50247a394034abd93b7c24,3,casa_construcao,esper instal,2,"[produt, vei, cheg, nao, praz, entreg, recom, ..."
219,edd86e0af4dc62b340c60d846643cbf9,5,moveis_escritorio,produt praz estend por entreg praz divulg prop...,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."
232,e47020cf52099bce9dc57524ea41218b,4,moveis_escritorio,indic loj compr sempr lannist,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."
...,...,...,...,...,...,...
35956,123381026f13c723cb967b14b1bc673a,5,moveis_escritorio,ador cade bonit conforta facil mont mesm mont ...,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."
36031,042b7c995635364a03c5bb79e994f513,4,fashion_roupa_masculina,filh gost,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."
36034,5d4beea8e2c71baee7f55c7ae3ae3313,1,moveis_escritorio,insatisfeit cade vei defeit fabr solt encaix p...,2,"[produt, vei, cheg, nao, praz, entreg, recom, ..."
36037,0cac8fef1bcba2a3335b132807520f6d,1,moveis_escritorio,produt vei fur err mont infeliz cade nao vai d...,0,"[bom, nao, produt, entreg, defeit, vei, cor, a..."


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [30]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
0    457
2    325
1    186
Name: count, dtype: int64

In [31]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[0, 2, 1]

In [32]:
[topic_word_mixture[i] for i in bad_frequency]

[['bom',
  'nao',
  'produt',
  'entreg',
  'defeit',
  'vei',
  'cor',
  'aind',
  'otim',
  'cheg'],
 ['produt',
  'vei',
  'cheg',
  'nao',
  'praz',
  'entreg',
  'recom',
  'compr',
  'boa',
  'unidad'],
 ['receb',
  'nao',
  'produt',
  'compr',
  'nao receb',
  'receb produt',
  'entreg',
  'outr',
  'doi',
  'ate']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [35]:

print("Mevcut sütunlar:")
print(df.columns.tolist())

print("\nToplam yorum sayısı:", len(df))

print("\nSatıcı bilgisi var mı?")
print("seller_id" in df.columns)

print("\nKâr bilgisi var mı?")
print("profits" in df.columns)


Mevcut sütunlar:
['review_id', 'length_review', 'review_score', 'order_id', 'product_category_name', 'full_review', 'full_review_cleaned', 'most_important_topic', 'most_important_words']

Toplam yorum sayısı: 36148

Satıcı bilgisi var mı?
False

Kâr bilgisi var mı?
False


In [36]:

# Satıcı verileri olmadığı için ürün kategorilerini analiz ediyoruz

category_analysis = df.groupby("product_category_name").agg(
    total_reviews=("review_score", "count"),
    average_score=("review_score", "mean"),
    bad_reviews=("review_score", lambda x: (x <= 3).sum())
)

category_analysis["bad_review_rate"] = (
    category_analysis["bad_reviews"] /
    category_analysis["total_reviews"] * 100
)

# En az 50 yorumu bulunan kategoriler
category_analysis = category_analysis[
    category_analysis["total_reviews"] >= 50
]

# Kötü yorum oranı en yüksek 10 kategori
worst_categories = category_analysis.sort_values(
    "bad_review_rate",
    ascending=False
).head(10)

worst_categories


,total_reviews,average_score,bad_reviews,bad_review_rate
product_category_name,,,,
moveis_escritorio,546,3.298535,261,47.802198
fashion_roupa_masculina,56,3.482143,21,37.500000
casa_construcao,177,3.672316,66,37.288136
audio,134,3.768657,49,36.567164
moveis_cozinha_area_de_servico_jantar_e_jardim,94,3.734043,34,36.170213
telefonia_fixa,96,3.697917,34,35.416667
moveis_sala,138,3.789855,48,34.782609
climatizacao,93,3.655914,32,34.408602
market_place,102,3.745098,35,34.313725


👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [38]:

# Mevcut veriyle en fazla kötü yorum alan kategorileri bul

category_analysis = df.groupby("product_category_name").agg(
    total_reviews=("review_score", "count"),
    average_score=("review_score", "mean"),
    bad_reviews=("review_score", lambda x: (x <= 3).sum())
)

category_analysis["bad_review_rate"] = (
    category_analysis["bad_reviews"] /
    category_analysis["total_reviews"] * 100
)

worst_categories = category_analysis[
    category_analysis["total_reviews"] >= 50
].sort_values(
    "bad_review_rate",
    ascending=False
).head(10)

worst_categories


,total_reviews,average_score,bad_reviews,bad_review_rate
product_category_name,,,,
moveis_escritorio,546,3.298535,261,47.802198
fashion_roupa_masculina,56,3.482143,21,37.500000
casa_construcao,177,3.672316,66,37.288136
audio,134,3.768657,49,36.567164
moveis_cozinha_area_de_servico_jantar_e_jardim,94,3.734043,34,36.170213
telefonia_fixa,96,3.697917,34,35.416667
moveis_sala,138,3.789855,48,34.782609
climatizacao,93,3.655914,32,34.408602
market_place,102,3.745098,35,34.313725


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [40]:

# En çok olumsuz yorum alan kategoriler
worst_category_names = worst_categories.index

category_bad_reviews = df[
    (df["product_category_name"].isin(worst_category_names)) &
    (df["review_score"] <= 3)
].copy()

print("Olumsuz yorum sayısı:", len(category_bad_reviews))

category_bad_reviews[
    ["product_category_name", "review_score",
     "most_important_topic"]
].head()


Olumsuz yorum sayısı: 637


,product_category_name,review_score,most_important_topic
45,moveis_escritorio,3,0
151,audio,1,2
212,casa_construcao,3,2
253,moveis_escritorio,1,2
294,moveis_escritorio,1,0


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [41]:

# Kötü yorum alan kategorilerin konu dağılımı
category_topics = pd.crosstab(
    category_bad_reviews["product_category_name"],
    category_bad_reviews["most_important_topic"]
)

category_topics


most_important_topic,0,1,2
product_category_name,,,
audio,16,14,19
casa_conforto,28,14,15
casa_construcao,8,33,25
climatizacao,9,20,3
fashion_roupa_masculina,4,12,5
market_place,11,15,9
moveis_cozinha_area_de_servico_jantar_e_jardim,9,12,13
moveis_escritorio,101,63,97
moveis_sala,11,16,21


In [42]:

# Her kategorinin en sık görülen şikâyet konusu
dominant_topics = category_topics.idxmax(axis=1)

print("Kategorilere göre baskın şikâyet konuları:")
print(dominant_topics)


Kategorilere göre baskın şikâyet konuları:
product_category_name
audio                                             2
casa_conforto                                     0
casa_construcao                                   1
climatizacao                                      1
fashion_roupa_masculina                           1
market_place                                      1
moveis_cozinha_area_de_servico_jantar_e_jardim    2
moveis_escritorio                                 0
moveis_sala                                       2
telefonia_fixa                                    0
dtype: int64


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [45]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [47]:

# En kötü ürün kategorilerindeki şikâyet konularını incele

category_topic_summary = (
    category_bad_reviews
    .groupby(["product_category_name", "most_important_topic"])
    .size()
    .reset_index(name="review_count")
)

category_topic_summary = category_topic_summary.sort_values(
    ["product_category_name", "review_count"],
    ascending=[True, False]
)

category_topic_summary.head(20)


,product_category_name,most_important_topic,review_count
2,audio,2,19
0,audio,0,16
1,audio,1,14
3,casa_conforto,0,28
5,casa_conforto,2,15
4,casa_conforto,1,14
7,casa_construcao,1,33
8,casa_construcao,2,25
6,casa_construcao,0,8
10,climatizacao,1,20


In [48]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [50]:

print("OLIST KÖTÜ YORUM ANALİZİ")
print("-" * 50)

print("Toplam yorum sayısı:", len(df))
print("Kötü yorum sayısı:", len(bad_reviews))

print("\nKötü yorum oranı: %{:.2f}".format(
    len(bad_reviews) / len(df) * 100
))

print("\nEn yüksek kötü yorum oranına sahip kategoriler:")
print(worst_categories)

print("\nKategorilere göre en önemli şikâyet konuları:")
print(
    top_complaints[
        ["product_category_name", "most_important_topic", "important_words"]
    ].to_string(index=False)
)



OLIST KÖTÜ YORUM ANALİZİ
--------------------------------------------------
Toplam yorum sayısı: 36148
Kötü yorum sayısı: 9658

Kötü yorum oranı: %26.72

En yüksek kötü yorum oranına sahip kategoriler:
                                                total_reviews  average_score  \
product_category_name                                                          
moveis_escritorio                                         546       3.298535   
fashion_roupa_masculina                                    56       3.482143   
casa_construcao                                           177       3.672316   
audio                                                     134       3.768657   
moveis_cozinha_area_de_servico_jantar_e_jardim             94       3.734043   
telefonia_fixa                                             96       3.697917   
moveis_sala                                               138       3.789855   
climatizacao                                               93       3.655914  

NameError: name 'top_complaints' is not defined

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.